# Multi-label Text Classification

Hệ thống Phân loại Văn bản Đa nhãn sử dụng kiến trúc **BERT + Bi-GRU + LSTM + CNN 1D**.

Tham khảo: *Predicting Job Titles from Job Descriptions with Multi-label Text Classification (arXiv:2112.11052)*.

## Bước 1: Thiết lập & Import

Cài đặt các thư viện cần thiết, import modules, và thiết lập device (GPU/CPU).

In [ ]:
# Cài đặt thư viện
!pip install transformers evaluate accelerate datasets -q

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score
)
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re
import random
import warnings
warnings.filterwarnings('ignore')

# --- Thiết lập Device ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

# --- Seed cho tính tái lập ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## Bước 2: Chuẩn bị & Tiền xử lý Dữ liệu

Load dataset **binhvq-news-corpus** từ HuggingFace, sample 20.000 mẫu.
Kết hợp `title + summary` làm text đầu vào, `category` làm nhãn.
Chuyển nhãn sang định dạng nhị phân đa nhãn bằng `MultiLabelBinarizer`.

In [ ]:
from datasets import load_dataset

# ============================================================
# Hyperparameters & Constants
# ============================================================
NUM_SAMPLES = 20000  # Số mẫu sample từ dataset gốc (14M rows)
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 5
BERT_LR = 2e-5
CLASSIFIER_LR = 1e-3
DROPOUT = 0.3
GRU_HIDDEN = 128
LSTM_HIDDEN = 128
CNN_OUT_CHANNELS = 64
KERNEL_SIZE = 3
THRESHOLD = 0.5
TRAIN_RATIO = 0.8
BERT_MODEL_NAME = 'bert-base-multilingual-cased'

# ============================================================
# Load dataset từ HuggingFace (STREAMING MODE)
# Không download toàn bộ dataset về disk — chỉ lấy đúng NUM_SAMPLES mẫu
# ============================================================
print(f'Loading {NUM_SAMPLES} samples via streaming...')
stream = load_dataset('ademax/binhvq-news-corpus', split='train', streaming=True)
stream = stream.shuffle(seed=SEED, buffer_size=10000)

# Lấy đúng NUM_SAMPLES mẫu, không tải thêm
data_list = []
for i, example in enumerate(tqdm(stream, total=NUM_SAMPLES, desc='Streaming')):
    if i >= NUM_SAMPLES:
        break
    data_list.append(example)

df = pd.DataFrame(data_list)
del data_list  # Giải phóng bộ nhớ
print(f'DataFrame shape: {df.shape}')
print(df.head(3))

# ============================================================
# Tạo cột text: kết hợp title + summary
# ============================================================
df['text'] = df['title'].fillna('') + ' . ' + df['summary'].fillna('')

# --- Loại bỏ các dòng thiếu category ---
df = df.dropna(subset=['category'])
df = df[df['category'].str.strip() != '']
print(f'After cleaning: {len(df)} rows')

# --- Chuyển category thành list (tương thích MultiLabelBinarizer) ---
df['labels'] = df['category'].apply(lambda x: [x.strip()])

# --- Xem danh sách categories ---
all_categories = sorted(df['category'].unique())
NUM_LABELS = len(all_categories)
print(f'Number of categories: {NUM_LABELS}')
print(f'Categories: {all_categories}')

# ============================================================
# Text Cleaning
# ============================================================
def clean_text(text):
    """Làm sạch văn bản tiếng Việt: lowercase, chuẩn hóa khoảng trắng.

    Args:
        text (str): Văn bản gốc.

    Returns:
        str: Văn bản đã làm sạch.
    """
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_text'] = df['text'].apply(clean_text)

# ============================================================
# Multi-label Binarization
# ============================================================
mlb = MultiLabelBinarizer()
label_matrix = mlb.fit_transform(df['labels'])
label_columns = mlb.classes_
print(f'Label classes: {list(label_columns)}')
print(f'Label matrix shape: {label_matrix.shape}')

# --- Train / Validation split ---
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['cleaned_text'].values, label_matrix,
    test_size=1 - TRAIN_RATIO, random_state=SEED,
    stratify=df['category'].values
)
print(f'Train: {len(train_texts)} | Validation: {len(val_texts)}')

# --- Phân bố nhãn ---
print('\nPhân bố nhãn trong dataset:')
label_counts = label_matrix.sum(axis=0)
for name, count in zip(label_columns, label_counts):
    print(f'  {name}: {int(count)}')

## Bước 3: Tokenization & Dataset Pipeline

Khởi tạo Tokenizer từ Hugging Face, viết custom Dataset class, và tạo DataLoader cho tập Train/Validation.

In [ ]:
# --- Khởi tạo Tokenizer ---
tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)
print(f'Tokenizer loaded: {BERT_MODEL_NAME}')

class MultiLabelDataset(Dataset):
    """Custom Dataset cho bài toán phân loại đa nhãn.

    Args:
        texts (np.ndarray): Mảng các văn bản đã được tiền xử lý.
        labels (np.ndarray): Ma trận nhãn nhị phân, shape [num_samples, num_labels].
        tokenizer (BertTokenizer): Tokenizer của BERT.
        max_len (int): Độ dài tối đa của chuỗi token.

    Returns (per __getitem__):
        dict: {
            'input_ids': tensor shape [max_len],
            'attention_mask': tensor shape [max_len],
            'labels': tensor shape [num_labels]
        }
    """

    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),       # [max_len]
            'attention_mask': encoding['attention_mask'].flatten(), # [max_len]
            'labels': torch.FloatTensor(label)                  # [num_labels]
        }

# --- Tạo Dataset ---
train_dataset = MultiLabelDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = MultiLabelDataset(val_texts, val_labels, tokenizer, MAX_LEN)

# --- Tạo DataLoader ---
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_dataloader)}')
print(f'Val batches: {len(val_dataloader)}')

# --- Kiểm tra 1 batch ---
sample_batch = next(iter(train_dataloader))
print(f"input_ids shape: {sample_batch['input_ids'].shape}")
print(f"attention_mask shape: {sample_batch['attention_mask'].shape}")
print(f"labels shape: {sample_batch['labels'].shape}")

## Bước 4: Xây dựng Kiến trúc Mô hình

Class `Bert_BiGRU_LSTM_CNN(nn.Module)` theo kiến trúc bài báo:
- BERT Embedding → Bi-GRU → LSTM → CNN 1D → Linear + Sigmoid

Chú thích tensor shape ở mỗi bước biến đổi.

In [ ]:
class Bert_BiGRU_LSTM_CNN(nn.Module):
    """Mô hình phân loại đa nhãn: BERT + Bi-GRU + LSTM + CNN 1D.

    Kiến trúc:
        BERT Embedding [batch, seq_len, 768]
        -> Bi-GRU [batch, seq_len, gru_hidden*2]
        -> LSTM [batch, seq_len, lstm_hidden*2]
        -> CNN 1D [batch, cnn_out_channels, seq_len]
        -> Global Max Pooling [batch, cnn_out_channels]
        -> Linear [batch, num_labels]

    Args:
        bert_model_name (str): Tên mô hình BERT pretrained.
        num_labels (int): Số lượng nhãn đầu ra.
        gru_hidden (int): Số hidden units của GRU mỗi chiều.
        lstm_hidden (int): Số hidden units của LSTM mỗi chiều.
        cnn_out_channels (int): Số kênh đầu ra của CNN.
        kernel_size (int): Kích thước kernel CNN.
        dropout (float): Tỷ lệ dropout.
    """

    def __init__(self, bert_model_name, num_labels, gru_hidden=128,
                 lstm_hidden=128, cnn_out_channels=64, kernel_size=3,
                 dropout=0.3):
        super().__init__()

        # --- Pre-trained BERT ---
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.bert_hidden_size = self.bert.config.hidden_size  # 768

        # --- Bi-GRU ---
        # Input:  [batch, seq_len, 768]
        # Output: [batch, seq_len, gru_hidden * 2]
        self.bigru = nn.GRU(
            input_size=self.bert_hidden_size,
            hidden_size=gru_hidden,
            batch_first=True,
            bidirectional=True
        )

        # --- LSTM ---
        # Input:  [batch, seq_len, gru_hidden * 2]
        # Output: [batch, seq_len, lstm_hidden * 2]
        self.lstm = nn.LSTM(
            input_size=gru_hidden * 2,
            hidden_size=lstm_hidden,
            batch_first=True,
            bidirectional=True
        )

        # --- CNN 1D ---
        # Input:  [batch, lstm_hidden * 2, seq_len] (sau permute)
        # Output: [batch, cnn_out_channels, seq_len']
        self.cnn = nn.Conv1d(
            in_channels=lstm_hidden * 2,
            out_channels=cnn_out_channels,
            kernel_size=kernel_size,
            padding=kernel_size // 2  # giữ chiều dài gần bằng
        )

        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

        # --- Output Layer ---
        # Input:  [batch, cnn_out_channels]
        # Output: [batch, num_labels]
        self.classifier = nn.Linear(cnn_out_channels, num_labels)

    def forward(self, input_ids, attention_mask):
        """
        Args:
            input_ids: [batch_size, seq_len]
            attention_mask: [batch_size, seq_len]

        Returns:
            logits: [batch_size, num_labels] (chưa qua Sigmoid)
        """
        # BERT: [batch, seq_len] -> [batch, seq_len, 768]
        bert_output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        hidden_states = bert_output.last_hidden_state  # [batch, seq_len, 768]

        # Bi-GRU: [batch, seq_len, 768] -> [batch, seq_len, gru_hidden*2]
        gru_out, _ = self.bigru(hidden_states)

        # LSTM: [batch, seq_len, gru_hidden*2] -> [batch, seq_len, lstm_hidden*2]
        lstm_out, _ = self.lstm(gru_out)

        # Permute cho CNN: [batch, seq_len, lstm_hidden*2] -> [batch, lstm_hidden*2, seq_len]
        cnn_input = lstm_out.permute(0, 2, 1)

        # CNN 1D: [batch, lstm_hidden*2, seq_len] -> [batch, cnn_out_channels, seq_len']
        cnn_out = self.relu(self.cnn(cnn_input))

        # Global Max Pooling: [batch, cnn_out_channels, seq_len'] -> [batch, cnn_out_channels]
        pooled = torch.max(cnn_out, dim=2)[0]

        # Dropout + Classifier
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)  # [batch, num_labels]

        return logits

# --- Khởi tạo mô hình ---
model = Bert_BiGRU_LSTM_CNN(
    bert_model_name=BERT_MODEL_NAME,
    num_labels=NUM_LABELS,
    gru_hidden=GRU_HIDDEN,
    lstm_hidden=LSTM_HIDDEN,
    cnn_out_channels=CNN_OUT_CHANNELS,
    kernel_size=KERNEL_SIZE,
    dropout=DROPOUT
).to(DEVICE)

# --- In thông tin mô hình ---
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

## Bước 5: Định nghĩa Loss Function & Optimizer

- **Loss:** `BCEWithLogitsLoss` (Binary Cross Entropy cho multi-label).
- **Optimizer:** `AdamW` với learning rate riêng biệt cho BERT (nhỏ) và classifier (lớn hơn).

In [ ]:
# --- Loss Function ---
# BCEWithLogitsLoss = Sigmoid + BCELoss (ổn định số học hơn)
criterion = nn.BCEWithLogitsLoss().to(DEVICE)

# --- Optimizer với Differential Learning Rate ---
# BERT layers dùng LR nhỏ để fine-tune nhẹ nhàng
# Classifier layers (GRU, LSTM, CNN, Linear) dùng LR lớn hơn
bert_params = list(model.bert.named_parameters())
classifier_params = (
    list(model.bigru.named_parameters()) +
    list(model.lstm.named_parameters()) +
    list(model.cnn.named_parameters()) +
    list(model.classifier.named_parameters())
)

optimizer = torch.optim.AdamW([
    {'params': [p for n, p in bert_params], 'lr': BERT_LR},
    {'params': [p for n, p in classifier_params], 'lr': CLASSIFIER_LR}
], weight_decay=0.01)

print(f'BERT LR: {BERT_LR}')
print(f'Classifier LR: {CLASSIFIER_LR}')
print(f'Loss: BCEWithLogitsLoss')
print(f'Optimizer: AdamW (differential LR)')

## Bước 6: Vòng lặp Huấn luyện (Training & Validation Loop)

- Thanh tiến trình `tqdm` cho mỗi epoch.
- `model.train()` / `model.eval()` đúng pha.
- Gradient clipping (`clip_grad_norm_`) trước `optimizer.step()`.
- `torch.cuda.empty_cache()` sau mỗi epoch.

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """Huấn luyện mô hình trong 1 epoch.

    Args:
        model: Mô hình Bert_BiGRU_LSTM_CNN.
        dataloader: DataLoader tập train.
        criterion: Loss function (BCEWithLogitsLoss).
        optimizer: Optimizer (AdamW).
        device: Device (cuda/cpu).

    Returns:
        float: Trung bình loss của epoch.
    """
    model.train()
    total_loss = 0
    progress_bar = tqdm(dataloader, desc='Training', leave=False)

    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)         # [batch, max_len]
        attention_mask = batch['attention_mask'].to(device) # [batch, max_len]
        labels = batch['labels'].to(device)                # [batch, num_labels]

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)  # [batch, num_labels]
        loss = criterion(logits, labels)
        loss.backward()

        # Gradient clipping để tránh exploding gradient (quan trọng cho RNN)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

    return total_loss / len(dataloader)


def validate_one_epoch(model, dataloader, criterion, device):
    """Đánh giá mô hình trên tập validation trong 1 epoch.

    Args:
        model: Mô hình Bert_BiGRU_LSTM_CNN.
        dataloader: DataLoader tập validation.
        criterion: Loss function.
        device: Device.

    Returns:
        tuple: (avg_loss, all_predictions, all_labels)
    """
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Validating', leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()

            # Áp dụng Sigmoid để chuyển logits -> xác suất [0, 1]
            preds = torch.sigmoid(logits)
            # Chuyển sang nhãn nhị phân dựa trên threshold
            preds = (preds >= THRESHOLD).int()

            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    return total_loss / len(dataloader), all_preds, all_labels


# ============================================================
# Main Training Loop
# ============================================================
print(f'Training for {EPOCHS} epochs...')
print('=' * 60)

best_f1 = 0
history = {'train_loss': [], 'val_loss': [], 'val_micro_f1': [], 'val_macro_f1': []}

for epoch in range(EPOCHS):
    print(f'\nEpoch {epoch + 1}/{EPOCHS}')
    print('-' * 40)

    # --- Training ---
    train_loss = train_one_epoch(model, train_dataloader, criterion, optimizer, DEVICE)

    # --- Validation ---
    val_loss, val_preds, val_true = validate_one_epoch(
        model, val_dataloader, criterion, DEVICE
    )

    # --- Metrics ---
    micro_f1 = f1_score(val_true, val_preds, average='micro', zero_division=0)
    macro_f1 = f1_score(val_true, val_preds, average='macro', zero_division=0)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_micro_f1'].append(micro_f1)
    history['val_macro_f1'].append(macro_f1)

    print(f'  Train Loss: {train_loss:.4f}')
    print(f'  Val Loss:   {val_loss:.4f}')
    print(f'  Micro F1:   {micro_f1:.4f}')
    print(f'  Macro F1:   {macro_f1:.4f}')

    # --- Lưu mô hình tốt nhất ---
    if micro_f1 > best_f1:
        best_f1 = micro_f1
        torch.save(model.state_dict(), 'best_model.pt')
        print(f'  -> Saved best model (Micro F1: {best_f1:.4f})')

    # --- Giải phóng bộ nhớ GPU ---
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print('\n' + '=' * 60)
print(f'Training complete! Best Micro F1: {best_f1:.4f}')

## Bước 7: Đánh giá Mô hình (Evaluation)

Tính toán các chỉ số: **Micro F1-score**, **Macro F1-score**, Precision và Recall.
In classification report chi tiết cho từng nhãn.

In [ ]:
# --- Load best model ---
model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))
print('Loaded best model weights.')

# --- Đánh giá trên tập Validation ---
val_loss, val_preds, val_true = validate_one_epoch(
    model, val_dataloader, criterion, DEVICE
)

# --- Tính các chỉ số ---
micro_f1 = f1_score(val_true, val_preds, average='micro', zero_division=0)
macro_f1 = f1_score(val_true, val_preds, average='macro', zero_division=0)
micro_precision = precision_score(val_true, val_preds, average='micro', zero_division=0)
macro_precision = precision_score(val_true, val_preds, average='macro', zero_division=0)
micro_recall = recall_score(val_true, val_preds, average='micro', zero_division=0)
macro_recall = recall_score(val_true, val_preds, average='macro', zero_division=0)

print('=' * 60)
print('EVALUATION RESULTS')
print('=' * 60)
print(f'Micro F1-score:  {micro_f1:.4f}')
print(f'Macro F1-score:  {macro_f1:.4f}')
print(f'Micro Precision: {micro_precision:.4f}')
print(f'Macro Precision: {macro_precision:.4f}')
print(f'Micro Recall:    {micro_recall:.4f}')
print(f'Macro Recall:    {macro_recall:.4f}')
print()

# --- Classification Report chi tiết ---
print('Per-label Classification Report:')
print('-' * 60)
report = classification_report(
    val_true, val_preds,
    target_names=list(label_columns),
    zero_division=0
)
print(report)

## Bước 8: Suy luận thử nghiệm (Inference)

Hàm `predict()` nhận một đoạn text mới và trả về danh sách các nhãn dự đoán vượt ngưỡng `threshold = 0.5`.

In [ ]:
def predict(text, model, tokenizer, mlb, device=DEVICE, threshold=THRESHOLD):
    """Dự đoán nhãn cho một đoạn văn bản mới.

    Args:
        text (str): Văn bản đầu vào cần phân loại.
        model (Bert_BiGRU_LSTM_CNN): Mô hình đã huấn luyện.
        tokenizer (BertTokenizer): Tokenizer.
        mlb (MultiLabelBinarizer): Bộ chuyển đổi nhãn.
        device (torch.device): Device.
        threshold (float): Ngưỡng xác suất (default=0.5).

    Returns:
        dict: {
            'predicted_labels': list[str],
            'probabilities': dict[str, float]
        }
    """
    model.eval()

    # Tiền xử lý text
    cleaned = clean_text(text)

    # Tokenize
    encoding = tokenizer(
        cleaned,
        add_special_tokens=True,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)         # [1, max_len]
    attention_mask = encoding['attention_mask'].to(device) # [1, max_len]

    with torch.no_grad():
        logits = model(input_ids, attention_mask)  # [1, num_labels]
        probs = torch.sigmoid(logits).cpu().numpy()[0]  # [num_labels]

    # Lọc nhãn vượt ngưỡng threshold
    predicted_indices = np.where(probs >= threshold)[0]
    predicted_labels = [mlb.classes_[i] for i in predicted_indices]

    # Xác suất từng nhãn
    prob_dict = {mlb.classes_[i]: round(float(probs[i]), 4) for i in range(len(mlb.classes_))}

    return {
        'predicted_labels': predicted_labels,
        'probabilities': prob_dict
    }


# ============================================================
# Thử nghiệm Inference
# ============================================================
test_texts = [
    'Apple chính thức ra mắt iPhone 16 với chip A18 mạnh mẽ và camera AI tiên tiến . Điện thoại mới có thiết kế hoàn toàn mới với màn hình Dynamic Island',
    'Đội tuyển Việt Nam giành chiến thắng lịch sử trước Thái Lan tại vòng loại World Cup 2026 với tỷ số 2-1 trên sân Mỹ Đình',
    'Ngân hàng Nhà nước tăng lãi suất cơ bản thêm 0.5 điểm phần trăm để kiềm chế lạm phát và ổn định tỷ giá đồng Việt Nam',
    'Bộ Giáo dục công bố phương án thi tốt nghiệp THPT 2026 với nhiều thay đổi quan trọng trong cách tính điểm xét tuyển đại học',
]

print('=' * 60)
print('INFERENCE DEMO')
print('=' * 60)

for i, text in enumerate(test_texts, 1):
    result = predict(text, model, tokenizer, mlb)
    print(f'\nTest {i}:')
    print(f'  Text: {text[:80]}...')
    print(f'  Predicted Labels: {result["predicted_labels"]}')
    print(f'  Probabilities:')
    for label, prob in sorted(result['probabilities'].items(), key=lambda x: -x[1]):
        marker = ' <<' if prob >= THRESHOLD else ''
        print(f'    {label:30s}: {prob:.4f}{marker}')